# NB68 — Matched FRH blending versus subspace steering

Two methods × WordNet/custom consensus sources × λ ∈ {0, 0.4}, with 1–4 steering units.
The full battery has **16 concepts, 60 named experiments, 74 directed configurations,
4,736 steered continuations, and eight shared baselines**. Both directions of each
counterfactual pair are separate runs; one pair counts as one steering unit.

The custom source is the consensus top 100 from the four files in `concepts/`.
Every output is evaluated against the same four fixed reference families:
WordNet frame/subspace and custom frame/subspace. Primary fluency cost is absolute
continuation-only ΔPPL. Native steering scores remain unchanged, so equal λ values
do not imply equal fluency pressure between methods.

Default execution is a **192-continuation smoke battery**, retaining all four unit
counts, both methods/sources/λ values, and both directions of a counterfactual at
each count. Set `SMOKE=False` for the full run; `RUN_GENERATION=False` inspects the
manifest and any existing results without loading the model. Outputs are resumable;
changed inputs, settings, construction, or implementation require a new prefix.


In [ ]:
# Papermill parameters
SMOKE = True
RUN_GENERATION = True
CONCEPTS_DIR = "concepts"
OUTPUT_PREFIX = ""

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from frames.experiments.comparison_report import write_report
from frames.experiments.steering_comparison import (
    ModelBackend,
    freeze_json,
    make_experiment,
    matrix_size,
    run_battery,
)

manifest = make_experiment(CONCEPTS_DIR, smoke=SMOKE)
prefix = OUTPUT_PREFIX or (
    "resources/68_comparison_smoke" if SMOKE else "resources/68_comparison"
)
Path(prefix).parent.mkdir(parents=True, exist_ok=True)
freeze_json(f"{prefix}_manifest.json", manifest)
print(
    f"{len(manifest['configs'])} directed configurations; "
    f"{matrix_size(manifest)} steered continuations; "
    f"{len(manifest['prompts'])} shared baselines"
)
display(
    pd.DataFrame(manifest["configs"])[
        ["id", "experiment_id", "unit_count", "atomic_count", "positive", "negative"]
    ]
)
display(pd.DataFrame(manifest["concepts"])[["concept", "synset", "words"]])

## Construction and generation

FRH uses equal-weight extrinsic frame blending. A counterfactual is first built as
`polar(F_A − F_B)` and then blended as one unit with the remaining frames.
Subspace steering blends the positive concepts to rank 8 and subtracts the repelled
concept's projection score. Both retain the greedy survivor rule.

The original eight-language WordNet token rows feed both builders unchanged.
Synset aliases are resolved against the loader's WordNet2021 corpus and recorded
alongside the exact requested JSON IDs. No concept is silently removed.

The 24-step CGD loop has one initial token plus 24 expansions, so baseline generation
uses `max_new_tokens=25`. Assistant prefixes and special tokens are excluded from
measurement. All raw continuation token counts, per-concept cosine values, and
baseline values are retained. Empty/unscorable outputs are flagged, never replaced
with successful zero-cost measurements.


In [ ]:
if RUN_GENERATION:
    backend = ModelBackend(manifest)
    records = run_battery(manifest, backend, prefix)
    print(f"Completed {len(records)} steered continuations")
else:
    print("Generation disabled; manifest inspection/reporting only")

## Matched comparisons

Keep the four reference families separate. Tables distinguish positive blends from
counterfactual cases and average pair directions within each named experiment.
The paired table compares methods at the same prompt, source, λ, and directed
configuration. Positive signed Δcos is desirable; negative ΔPPL is more fluent.
Minimum signed Δcos and the all-components-improved rate prevent a strong component
from hiding another component's failure. Attraction and repulsion are also retained
separately. This fixed battery is illustrative, not a balanced sample of all combinations.


In [ ]:
if Path(f"{prefix}.jsonl").exists():
    write_report(prefix)
    display(pd.read_csv(f"{prefix}_summary.csv"))
    display(pd.read_csv(f"{prefix}_paired.csv").head(20))
    display(Image(filename=f"{prefix}_cosine_ppl.png"))
    print(f"Gallery: {prefix}_gallery.md")
else:
    print("No generated results yet")